## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

In [ ]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr

In [ ]:
# show the interpreter the notebook is actually using
import sys
print(sys.executable)
print(sys.path[:3])

In [ ]:
import langchain, os, pkgutil
print("langchain:", langchain.__file__)
print("version:", getattr(langchain, "__version__", "unknown"))
print("top-level submodules:", [m.name for m in pkgutil.iter_modules([os.path.dirname(langchain.__file__)])])

In [ ]:
# imports for langchain (robust fallback)
try:
    # try multiple common import styles
    try:
        from langchain.document_loaders.directory import DirectoryLoader
        from langchain.document_loaders.text import TextLoader
    except Exception:
        from langchain.document_loaders import DirectoryLoader, TextLoader  # older layouts
    from langchain.text_splitter import CharacterTextSplitter
except Exception as e:
    print("langchain loaders not available (using local fallback):", e)
    from typing import List, NamedTuple

    # Minimal Document replacement (self-contained)
    class Document(NamedTuple):
        page_content: str
        metadata: dict

    class TextLoader:
        def __init__(self, path, encoding='utf-8'):
            self.path = path
            self.encoding = encoding

        def load(self) -> List[Document]:
            with open(self.path, 'r', encoding=self.encoding) as f:
                return [Document(page_content=f.read(), metadata={"source": self.path})]

    class DirectoryLoader:
        def __init__(self, folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=None):
            self.folder = folder
            self.glob = glob
            self.loader_cls = loader_cls
            self.loader_kwargs = loader_kwargs or {}

        def load(self) -> List[Document]:
            import glob as _glob, os as _os
            pattern = _os.path.join(self.folder, self.glob)
            files = _glob.glob(pattern, recursive=True)
            docs: List[Document] = []
            for p in files:
                loader = self.loader_cls(p, **self.loader_kwargs)
                docs.extend(loader.load())
            return docs

    class CharacterTextSplitter:
        def __init__(self, chunk_size=1000, chunk_overlap=200):
            self.chunk_size = chunk_size
            self.chunk_overlap = chunk_overlap

        def split_documents(self, documents: List[Document]) -> List[Document]:
            out: List[Document] = []
            for d in documents:
                text = d.page_content or ""
                i = 0
                step = max(1, self.chunk_size - self.chunk_overlap)
                while i < len(text):
                    chunk = text[i:i + self.chunk_size]
                    meta = dict(getattr(d, "metadata", {}))
                    out.append(Document(page_content=chunk, metadata=meta))
                    i += step
            return out

In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_db"

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [ ]:
# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase
# Thank you Mark D. and Zoya H. for fixing a bug here..

folders = glob.glob("knowledge-base/*")

# With thanks to CG and Jon R, students on the course, for this fix needed for some users 
text_loader_kwargs = {'encoding': 'utf-8'}
# If that doesn't work, some Windows users might need to uncomment the next line instead
# text_loader_kwargs={'autodetect_encoding': True}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

In [ ]:
len(documents)

In [ ]:
documents[24]

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [ ]:
len(chunks)

In [ ]:
chunks[6]

In [ ]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

In [ ]:
for chunk in chunks:
    if 'CEO' in chunk.page_content:
        print(chunk)
        print("_________")